### Imports / setup

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics

from mlxtend.plotting import plot_confusion_matrix
from mlxtend.plotting import plot_decision_regions

from matplotlib import pyplot as plt

from pathlib import Path

import tensorflow
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras import optimizers, regularizers
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import plot_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, Callback

print(tensorflow.__version__)


### Charger les données

In [ ]:
df = pd.read_csv('./dataset/labels.csv')
df["filename"] = df["id"] + ".jpg"
train_dir = "dataset/train"
df = df[df["filename"].apply(lambda f: os.path.exists(os.path.join(train_dir, f)))].copy()
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["breed"], random_state=42)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")
train_df.head()


### Determiné Dense output

In [ ]:
df["breed"].unique().__len__()

### Créer le modèle

In [ ]:
os.makedirs("model", exist_ok=True)

# Rotations, zooms, flips pour mieux généraliser
datagen_train = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode="nearest",
    brightness_range=[0.8, 1.2],
    channel_shift_range=20
)

datagen_val = ImageDataGenerator(preprocessing_function=mobilenet_preprocess)

train_gen = datagen_train.flow_from_dataframe(
    train_df, directory=train_dir, x_col="filename", y_col="breed",
    target_size=(224, 224), batch_size=32, class_mode="categorical", shuffle=True, seed=42,
)
val_gen = datagen_val.flow_from_dataframe(
    val_df, directory=train_dir, x_col="filename", y_col="breed",
    target_size=(224, 224), batch_size=32, class_mode="categorical", shuffle=False,
)

base = MobileNetV2(input_shape=(224, 224, 3), weights="imagenet", include_top=False, pooling=None)
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(1024, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

x = BatchNormalization()(x)
x = Dropout(0.4)(x)

outputs = Dense(120, activation="softmax")(x)
model = Model(inputs=base.input, outputs=outputs, name="dog_breed_model")
model.compile(optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

### Entraînement

In [ ]:
# Sauvegarde le meilleur modèle basé sur val_accuracy
model_checkpoint = ModelCheckpoint(
    "model/model_1024.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)
callbacks = [model_checkpoint]

In [ ]:
EPOCHS = 20
STEPS = max(1, len(train_df) // 32)
VAL_STEPS = max(1, len(val_df) // 32)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    steps_per_epoch=STEPS,
    validation_steps=VAL_STEPS,
    callbacks=callbacks,
    verbose=1,
)

### Tests du modèle

In [ ]:
val_gen.reset()
loss, accuracy = model.evaluate(val_gen, steps=VAL_STEPS, verbose=1)
print(f"Validation Loss: {loss:.4f}, Validation Accuracy: {accuracy:.4f}")

# S'assurer que model/labels.json a le même ordre que les classes du générateur
classes = [c for c, _ in sorted(train_gen.class_indices.items(), key=lambda x: x[1])]
with open("model/labels.json", "w", encoding="utf-8") as f:
    json.dump(classes, f, indent=2, ensure_ascii=False)
print("Modèle et labels enregistrés.")